In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl accelerate

import os, json, re, time, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
import emoji
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.0 MB/s eta 0:00:00
Device: cuda


In [2]:
REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

with open(os.path.join(DOCS_PATH, "patterns.json"), encoding="utf-8") as f:
    pattern_dict = json.load(f)
with open(os.path.join(DOCS_PATH, "emojis.json"), encoding="utf-8") as f:
    emoji_dict = json.load(f)

teen_dict = {}
with open(os.path.join(DOCS_PATH, "teencode4.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "\t" in line:
            old, new = line.split("\t", 1)
            teen_dict[old] = new

excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
excel_file = pd.ExcelFile(excel_path)
if "train" in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name="train")
    val_df   = pd.read_excel(excel_file, sheet_name="val")
    test_df  = pd.read_excel(excel_file, sheet_name="test")
else:
    df = pd.read_excel(excel_file, sheet_name="Sheet1")
    train_df = df[df["set"] == "train"].copy()
    val_df   = df[df["set"] == "val"].copy()
    test_df  = df[df["set"] == "test"].copy()

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Cloning into '/kaggle/working/ViGoEmotions_Original'...


Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


In [3]:
def normalize_pattern(text):
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern=pattern, repl=replacement, string=text)
    return text

def remove_duplicate_chars(text):
    prev_char, result = None, []
    for char in text:
        if char.isalpha() and prev_char == char:
            continue
        prev_char = char
        result.append(char)
    return "".join(result)

def remove_duplicate_emoji(text):
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji:
                continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    return "".join(result)

def replace_teencode(text):
    for old_word, new_word in teen_dict.items():
        pattern = re.compile(r"\b{}\b".format(re.escape(old_word)))
        text = pattern.sub(new_word, text)
    return text

def clean_text(text):
    """S1 theo thầy: KHÔNG gọi replacing_emojis"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = normalize_pattern(text)
    text = remove_duplicate_chars(text)
    text = remove_duplicate_emoji(text)
    text = replace_teencode(text)
    # text = replacing_emojis(text)  # ← thầy COMMENT, S1 không dùng

    text = re.sub(r"(?<![.,!?;:])\n", r". ", text)
    text = re.sub(r"\n([.,!?;:])?", r" \1", text)
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Applying S1 preprocessing (no emoji replace)...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("Done:", train_df["text"].iloc[0])

Applying S1 preprocessing (no emoji replace)...
Done: xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


In [4]:
label_dict_path = os.path.join(DOCS_PATH, "label_dict.json")
if not os.path.exists(label_dict_path):
    label_dict_path = os.path.join(CORPUS_PATH, "label_dict.json")

with open(label_dict_path, encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Labels:", len(label_dict))

def encode_labels(label_str, label_dict):
    labels = str(label_str).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)
    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]
print("Labels encoded")

Labels: 28
Labels encoded


In [5]:
!pip install -q sentencepiece protobuf huggingface_hub

from huggingface_hub import snapshot_download
from transformers import T5Tokenizer
import sentencepiece as spm

model_name = "VietAI/vit5-base"

local_dir = snapshot_download(
    repo_id=model_name,
    allow_patterns=["spiece.model", "tokenizer_config.json", "special_tokens_map.json", "config.json"]
)
print("Local dir:", local_dir)

sp = spm.SentencePieceProcessor()
sp.load(f"{local_dir}/spiece.model")
print("SentencePiece pieces:", sp.get_piece_size()) 

# Load tokenizer từ thư mục local
tokenizer = T5Tokenizer.from_pretrained(local_dir)
tokenizer.name_or_path = model_name

print("Vocab size:", tokenizer.vocab_size)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Local dir: /root/.cache/huggingface/hub/models--VietAI--vit5-base/snapshots/2209a38d735ede63e88f5aa52bcdc11a05a37b85
SentencePiece pieces: 36000
Vocab size: 36096


In [6]:
from pyvi.ViTokenizer import tokenize

model_type = "vit5"
model_name = "VietAI/vit5-base"
max_len = 200
BATCH_SIZE = 8

tokenizer.name_or_path = model_name 

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=200):
        self.texts = texts
        self.labels = torch.tensor(np.array(labels), dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prefix = "classification: "  

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        text = tokenize(text)
        text = self.prefix + text

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader ready | batches:", len(train_loader))

DataLoader ready | batches: 2067


In [7]:
from transformers import T5EncoderModel

In [8]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_type="vit5"):
        super().__init__()
        self.model_type = model_type
        model_name = "VietAI/vit5-base"

        self.backbone = T5EncoderModel.from_pretrained(model_name)
        self.drop = nn.Dropout(0.2)
        hidden = getattr(self.backbone.config, "hidden_size", None) or self.backbone.config.d_model
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        pooled = outputs.last_hidden_state[:, 0, :]
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(n_classes=len(label_dict), model_type=model_type).to(device)
print("ViT5 (T5EncoderModel) loaded - S3")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: VietAI/vit5-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ViT5 (T5EncoderModel) loaded - S3
Parameters: 140,419,228


In [9]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            if is_train:
                optimizer.zero_grad()
            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())
    y, p = np.vstack(all_y), np.vstack(all_p)
    return np.mean(losses), f1_score(y, p, average="macro", zero_division=0)

best_f1 = 0.0
history = {"epoch": [], "train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training BARTpho (S1)...")
for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_f1 = run_epoch(model, train_loader, True)
    val_loss, val_f1 = run_epoch(model, val_loader, False)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_s1_best.pth")
        print(f"Saved best (Val F1={best_f1:.4f})")

pd.DataFrame(history).to_excel(f"/kaggle/working/reports/metrics_{model_type}_s1.xlsx", index=False)
print("Metrics saved:", f"/kaggle/working/reports/metrics_{model_type}_s1.xlsx")

Start training BARTpho (S1)...

Epoch 1/12


  0%|          | 2/2067 [00:02<33:55,  1.01it/s]  

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Train Loss: 1.1687 | Train F1: 0.1795
Val   Loss: 0.8883 | Val   F1: 0.3439
Saved best (Val F1=0.3439)

Epoch 2/12


Train Loss: 0.8355 | Train F1: 0.3449
Val   Loss: 0.7941 | Val   F1: 0.3875
Saved best (Val F1=0.3875)

Epoch 3/12


...

Train Loss: 0.2412 | Train F1: 0.7240
Val   Loss: 1.2686 | Val   F1: 0.5048
Saved best (Val F1=0.5048)

Epoch 9/12


Train Loss: 0.2014 | Train F1: 0.7663
Val   Loss: 1.3585 | Val   F1: 0.5155
Saved best (Val F1=0.5155)

Epoch 10/12


Train Loss: 0.1686 | Train F1: 0.7991
Val   Loss: 1.5021 | Val   F1: 0.5203
Saved best (Val F1=0.5203)

Epoch 11/12


Train Loss: 0.1432 | Train F1: 0.8269
Val   Loss: 1.5401 | Val   F1: 0.5301
Saved best (Val F1=0.5301)

Epoch 12/12


Train Loss: 0.1244 | Train F1: 0.8478
Val   Loss: 1.5863 | Val   F1: 0.5314
Saved best (Val F1=0.5314)
Metrics saved: /kaggle/working/reports/metrics_vit5_s1.xlsx


In [10]:
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_s1_best.pth"))
model.eval()

all_targets, all_preds = [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)
        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
print(f"\nTEST Macro F1: {macro_f1:.4f}")
print(f"Test Micro F1: {micro_f1:.4f}")

report = classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0, output_dict=True
)
pd.DataFrame(report).transpose().to_excel(
    f"/kaggle/working/reports/classification_report_{model_type}_s1.xlsx",
    index=True
)
print(classification_report(y_true, y_pred, target_names=list(label_dict.values()), zero_division=0))
print("Report saved:", f"/kaggle/working/reports/classification_report_{model_type}_s1.xlsx")

100%|██████████| 259/259 [00:25<00:00,  9.97it/s]



TEST Macro F1: 0.5317
Test Micro F1: 0.5416
                precision    recall  f1-score   support

     amusement       0.58      0.74      0.65       374
    excitement       0.39      0.61      0.48        98
           joy       0.40      0.67      0.50       204
          love       0.47      0.77      0.58       143
        desire       0.38      0.59      0.46        80
      optimism       0.54      0.73      0.62       142
        caring       0.49      0.77      0.60       150
         pride       0.62      0.66      0.64        86
    admiration       0.43      0.62      0.51       101
     gratitude       0.69      0.84      0.76       108
        relief       0.34      0.63      0.44        60
      approval       0.44      0.60      0.51       115
   realization       0.37      0.46      0.41        95
      surprise       0.44      0.52      0.47        85
     curiosity       0.55      0.62      0.58       100
     confusion       0.36      0.45      0.40        84
  